# **Semana 10: Big Data - Databricks Community Edition y PySpark (NB1 - Conceptual)**

## **Propósito de la Sesión**
Introducir entornos de Big Data con Apache Spark utilizando la plataforma gratuita Databricks Community Edition. Aprenderemos a crear un cluster, subir datos, leer archivos CSV con PySpark y comparar su rendimiento con pandas en datasets de gran tamaño.

### **Objetivos de Aprendizaje**
Al finalizar este notebook, el estudiante será capaz de:
1. **Crear** una cuenta gratuita en Databricks Community Edition y configurar un cluster.
2. **Subir** datos a Databricks utilizando comandos DBFS (Databricks File System).
3. **Leer** archivos CSV con PySpark y mostrar su esquema.
4. **Comparar** el rendimiento de PySpark vs pandas en el procesamiento de millones de filas.
5. **Comprender** las 5 V del Big Data y la importancia del procesamiento distribuido.

## **Configuración Inicial**

### **Nota importante:**
Este notebook está diseñado para ejecutarse en **Databricks Community Edition**. Sin embargo, incluimos código que también puede ejecutarse en Google Colab con PySpark instalado para fines de práctica. Las instrucciones detalladas para Databricks se proporcionan en texto.

### **Parte 1: Creación de cuenta y cluster en Databricks CE**

Sigue estos pasos para configurar tu entorno gratuito:

1. **Registro**:
   *   Ve a [https://community.cloud.databricks.com/](https://community.cloud.databricks.com/).
   *   Haz clic en "Sign up" (Registrarse).
   *   Completa el formulario con tu correo electrónico (puede ser personal o institucional).
   *   Espera el correo de confirmación y sigue el enlace para activar tu cuenta.

2. **Inicio de sesión**:
   *   Una vez activada, inicia sesión en [https://community.cloud.databricks.com/](https://community.cloud.databricks.com/).

3. **Creación del cluster**:
   *   En el panel izquierdo, haz clic en **Compute** (o **Clusters**).
   *   Haz clic en el botón **Create Cluster**.
   *   Asigna un nombre al cluster (ej. "Cluster-Sesion10").
   *   Selecciona la versión de runtime (elige la más reciente, por ejemplo, **11.3 LTS**).
   *   En "Access Mode", deja la opción por defecto (Single User).
   *   **Importante**: Databricks CE ofrece un cluster gratuito de un solo nodo con 15 GB de memoria y 2 núcleos.
   *   Haz clic en **Create Cluster**.

4. **Espera**: La creación del cluster toma 2-3 minutos. Verás que el estado cambia de "Pending" a "Running".

✅ ¡Ya tienes tu cluster de Big Data listo para usar!

### **Parte 2: Subir datos a Databricks (DBFS)**

Databricks utiliza **DBFS (Databricks File System)** para almacenar archivos. Subiremos un dataset de ejemplo.

**Instrucciones en Databricks:**

1.  En el panel izquierdo, haz clic en **Data** (icono de base de datos).
2.  Haz clic en el botón **Add Data**.
3.  Selecciona la pestaña **DBFS**.
4.  Haz clic en **Browse DBFS** y luego en el botón **Upload** (subir archivo).
5.  Sube el archivo CSV que usarás. Si no tienes uno, puedes usar el dataset de ventas online de la sesión anterior.

**Alternativa: Subir por comando (en un notebook de Databricks):**

Puedes usar comandos mágicos para subir archivos directamente desde una URL:

```python
# En una celda de Databricks (no en Colab)
import urllib.request
urllib.request.urlretrieve("https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx", "/tmp/online_retail.xlsx")
dbutils.fs.mv("file:/tmp/online_retail.xlsx", "dbfs:/FileStore/tables/online_retail.xlsx")
```

Para este notebook conceptual, **simularemos la lectura en Colab** con un dataset generado sintéticamente.

In [ ]:
# Instalación de PySpark en Colab (solo para simulación)
!pip install pyspark --quiet
!pip install pandas numpy --quiet

# Importaciones
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de PySpark
from pyspark.sql import SparkSession

# Inicializar SparkSession (en Databricks esto ya está hecho)
spark = SparkSession.builder \
    .appName("BigData_Sesion10") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

print(f"Spark versión: {spark.version}")
print(f"Spark UI disponible en: {spark.sparkContext.uiWebUrl}")

# Configuración de visualización
sns.set_style("whitegrid")
pd.set_option('display.max_columns', None)

### **Parte 3: Generación de Dataset Masivo para Pruebas**

Para simular un escenario de Big Data, generaremos un dataset con **10 millones de filas**. Este dataset representará transacciones de ventas.

In [ ]:
print("--- GENERACIÓN DE DATASET MASIVO ---")

# Parámetros
num_rows = 10_000_000  # 10 millones de filas
num_unique_products = 1000
num_countries = 20

print(f"Generando {num_rows:,} filas de datos...")

start_gen = time.time()

# Semilla para reproducibilidad
np.random.seed(42)

# Generar datos
data = {
    'transaction_id': range(1, num_rows + 1),
    'product_id': np.random.randint(1, num_unique_products + 1, num_rows),
    'quantity': np.random.randint(1, 10, num_rows),
    'unit_price': np.round(np.random.uniform(5, 500, num_rows), 2),
    'country': np.random.choice([f'Country_{i}' for i in range(1, num_countries + 1)], num_rows),
    'timestamp': pd.date_range(start='2023-01-01', periods=num_rows, freq='s').strftime('%Y-%m-%d %H:%M:%S')
}

# Crear DataFrame de pandas
df_pandas = pd.DataFrame(data)
df_pandas['total_amount'] = df_pandas['quantity'] * df_pandas['unit_price']

gen_time = time.time() - start_gen
print(f"Dataset generado en {gen_time:.2f} segundos.")
print(f"Tamaño en memoria: {df_pandas.memory_usage(deep=True).sum() / 1024**3:.2f} GB")

print("\nPrimeras 5 filas del dataset:")
display(df_pandas.head())

### **Parte 4: Lectura de CSV con PySpark y Mostrar Schema**

Primero guardaremos el dataset como CSV para simular la lectura desde archivo.

In [ ]:
# Guardar como CSV (esto puede tomar unos segundos)
csv_path = 'ventas_masivas.csv'
df_pandas.to_csv(csv_path, index=False)
print(f"Dataset guardado en '{csv_path}'")

# Leer con PySpark
print("\n--- LECTURA CON PYSPARK ---")
start_spark_read = time.time()

# Leer CSV con inferSchema para que detecte tipos automáticamente
df_spark = spark.read.option("header", "true") \
                     .option("inferSchema", "true") \
                     .csv(csv_path)

spark_read_time = time.time() - start_spark_read

# Mostrar esquema
print("\nEsquema del DataFrame (PySpark):")
df_spark.printSchema()

# Mostrar primeras filas
print("\nPrimeras 5 filas:")
df_spark.show(5, truncate=False)

print(f"\nTiempo de lectura con PySpark: {spark_read_time:.2f} segundos")
print(f"Número de particiones: {df_spark.rdd.getNumPartitions()}")

### **Parte 5: Comparación de Rendimiento - PySpark vs pandas**

Realizaremos una operación común en análisis de datos: **calcular el total de ventas por país**. Compararemos el tiempo de ejecución entre pandas y PySpark.

In [ ]:
print("--- COMPARACIÓN DE RENDIMIENTO ---")

# Operación: ventas totales por país

# 1. pandas
print("\n📊 Ejecutando con pandas...")
start_pandas = time.time()

pandas_result = df_pandas.groupby('country')['total_amount'].sum().reset_index()
pandas_result = pandas_result.sort_values('total_amount', ascending=False)

pandas_time = time.time() - start_pandas
print(f"✅ pandas completado en {pandas_time:.2f} segundos")

# 2. PySpark
print("\n⚡ Ejecutando con PySpark...")
start_spark = time.time()

from pyspark.sql.functions import sum as spark_sum

spark_result = df_spark.groupBy('country').agg(spark_sum('total_amount').alias('total_amount'))
spark_result = spark_result.orderBy('total_amount', ascending=False)

# Forzar la ejecución (acción)
spark_result_collected = spark_result.collect()

spark_time = time.time() - start_spark
print(f"✅ PySpark completado en {spark_time:.2f} segundos")

# Mostrar resultados
print("\nTop 5 países por ventas (pandas):")
display(pandas_result.head())

print("\nTop 5 países por ventas (PySpark):")
display(spark_result.limit(5).toPandas())

# Comparación visual
plt.figure(figsize=(8, 5))
methods = ['pandas', 'PySpark']
times = [pandas_time, spark_time]
colors = ['skyblue', 'coral']

plt.bar(methods, times, color=colors)
plt.ylabel('Tiempo (segundos)')
plt.title(f'Comparación de Rendimiento: pandas vs PySpark\n({num_rows:,} filas)')
for i, v in enumerate(times):
    plt.text(i, v + 0.5, f"{v:.2f}s", ha='center', fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.show()

speedup = pandas_time / spark_time
print(f"\n📈 PySpark es {speedup:.2f} veces más rápido que pandas en esta operación.")

print("\n**Explicación:**")
print("- pandas procesa en un solo núcleo, limitado por la memoria de la máquina.")
print("- PySpark distribuye la carga en múltiples núcleos (y nodos en un cluster real), permitiendo procesar datasets que no caben en memoria.")
print("- En Databricks CE, aunque es un solo nodo, Spark aprovecha los múltiples núcleos y optimiza las operaciones.")

### **Parte 6: Exploración de las 5 V del Big Data con nuestro ejemplo**

Ahora relacionaremos nuestro ejemplo con las **5 V del Big Data**:

1.  **Volumen**: Hemos trabajado con 10 millones de filas (~1-2 GB). En Big Data real, son terabytes o petabytes.
2.  **Velocidad**: Nuestros datos tienen timestamps cada segundo. En streaming real, llegan miles por segundo.
3.  **Variedad**: Nuestros datos son estructurados (CSV). En Big Data, también hay JSON, imágenes, texto.
4.  **Veracidad**: Nuestros datos son sintéticos y "limpios". En la realidad, hay que lidiar con datos sucios y faltantes.
5.  **Valor**: La agregación por país nos da valor de negocio (¿qué países generan más ingresos?).

In [ ]:
# Mostrar algunas estadísticas de las 5 V
print("--- LAS 5 V DEL BIG DATA EN NUESTRO EJEMPLO ---")

# Volumen
print(f"\n📦 VOLUMEN:")
print(f"  - Filas: {num_rows:,}")
print(f"  - Tamaño en memoria: {df_pandas.memory_usage(deep=True).sum() / 1024**3:.2f} GB")

# Velocidad
print(f"\n⚡ VELOCIDAD:")
print(f"  - Frecuencia de datos: 1 transacción por segundo")
print(f"  - Transacciones por día: 86,400")
print(f"  - Días simulados: {num_rows / 86400:.1f} días")

# Variedad
print(f"\n🔀 VARIEDAD:")
print(f"  - Tipos de datos: entero, flotante, string, timestamp")
print(f"  - Productos únicos: {num_unique_products}")
print(f"  - Países únicos: {num_countries}")

# Veracidad (simulación)
print(f"\n✅ VERACIDAD:")
# Introducimos algunos valores nulos para simular datos reales
df_pandas_with_nulls = df_pandas.copy()
null_indices = np.random.choice(df_pandas.index, size=1000, replace=False)
df_pandas_with_nulls.loc[null_indices, 'country'] = None
null_percentage = (df_pandas_with_nulls['country'].isna().sum() / num_rows) * 100
print(f"  - Porcentaje de datos faltantes (simulado): {null_percentage:.3f}%")

# Valor
print(f"\n💰 VALOR:")
top_country = pandas_result.iloc[0]
print(f"  - País con mayores ventas: {top_country['country']} (${top_country['total_amount']:,.2f})")
print(f"  - Total de ventas en el dataset: ${df_pandas['total_amount'].sum():,.2f}")

### **Parte 7: Ejemplo de operaciones comunes en PySpark**

Veamos algunas operaciones típicas que se realizan en PySpark para análisis de Big Data.

In [ ]:
print("--- OPERACIONES COMUNES EN PYSPARK ---")

# 1. Filtrado
print("\n1. Filtrar ventas con cantidad > 5 unidades:")
filtered_df = df_spark.filter(df_spark['quantity'] > 5)
print(f"   Registros originales: {df_spark.count():,}")
print(f"   Registros filtrados: {filtered_df.count():,}")

# 2. Selección de columnas
print("\n2. Seleccionar solo transaction_id, country y total_amount:")
selected_df = df_spark.select('transaction_id', 'country', 'total_amount')
selected_df.show(5)

# 3. Agregación múltiple
print("\n3. Estadísticas por producto:")
from pyspark.sql.functions import avg, min, max, count

product_stats = df_spark.groupBy('product_id').agg(
    count('*').alias('num_transacciones'),
    avg('total_amount').alias('monto_promedio'),
    min('total_amount').alias('monto_minimo'),
    max('total_amount').alias('monto_maximo')
).orderBy('num_transacciones', ascending=False)

print("Top 5 productos por número de transacciones:")
product_stats.show(5)

# 4. Ordenamiento
print("\n4. Top 5 transacciones por monto:")
df_spark.orderBy('total_amount', ascending=False).select('transaction_id', 'product_id', 'total_amount').show(5)

---
## **Ejercicios Prácticos para el Estudiante**

Ahora aplica lo aprendido con estos ejercicios basados en PySpark.

### **Ejercicio 1: Exploración de datos con PySpark**

Usando el DataFrame `df_spark`:
1.  Muestra el número total de filas.
2.  Calcula el precio promedio (`unit_price`) por país.
3.  Encuentra el producto (`product_id`) con la mayor cantidad total vendida (`quantity`).

Muestra los resultados de manera clara.

In [ ]:
# --- INICIO DE TU CÓDIGO ---

# 1. Total de filas
total_filas = df_spark.count()
print(f"Total de filas: {total_filas:,}")

# 2. Precio promedio por país
# ...

# 3. Producto con mayor cantidad vendida
# ...

# --- FIN DE TU CÓDIGO ---

### **Ejercicio 2: Comparación con pandas en otra operación**

Elige otra operación (por ejemplo, calcular el promedio de `total_amount` por `product_id`) y compara el tiempo de ejecución entre pandas y PySpark. ¿Cuál es más rápido? ¿Por qué?

In [ ]:
# --- INICIO DE TU CÓDIGO ---

# Operación elegida: promedio de total_amount por product_id

# pandas
# ...

# PySpark
# ...

# Comparación
# ...

# --- FIN DE TU CÓDIGO ---

### **Ejercicio 3: Análisis de las 5 V en un dataset real**

Investiga y describe brevemente cómo se aplican las 5 V del Big Data en uno de los siguientes casos reales (elige uno):
*   Twitter (análisis de trending topics)
*   Netflix (recomendaciones de películas)
*   Uber (coordinación de viajes en tiempo real)
*   Un banco (detección de fraudes)

Escribe tu respuesta en una celda Markdown.

---
## **Conclusiones**

En esta sesión hemos:
1.  Configurado Databricks Community Edition y creado nuestro primer cluster.
2.  Aprendido a subir datos a Databricks y leerlos con PySpark.
3.  Comparado el rendimiento de PySpark vs pandas en un dataset de 10 millones de filas.
4.  Explorado las 5 V del Big Data con ejemplos concretos.
5.  Realizado operaciones comunes de análisis con PySpark.

**Conceptos clave:**
*   **Big Data** no es solo volumen, sino velocidad, variedad, veracidad y valor.
*   **PySpark** permite procesar datos que no caben en una sola máquina mediante procesamiento distribuido.
*   **Databricks** ofrece una plataforma unificada para trabajar con Spark de manera sencilla.

En el próximo notebook, aplicaremos estos conocimientos en ejercicios prácticos con datasets reales.

In [ ]:
# Cerrar SparkSession (opcional, en Databricks no es necesario)
spark.stop()